# 09. Análisis de Errores del Modelo Híbrido

**Objetivo:** caracterizar errores por clase y generar evidencia cualitativa/cuantitativa para la discusión metodológica.
**Entradas (inputs):** `data/outputs/cierre_modelos_dev_<timestamp>/decision_modelo_final.json`, `ranking_modelos_dev.csv`, predicciones del `train_<run_id>` de referencia y `data/splits/dataset_base.csv`.
**Salidas (outputs):** `data/outputs/error_analysis_<run_id>/resumen_error_por_clase.csv`, `matriz_confusion.csv`, `terminos_distintivos_errores.csv`, `casos_mal_clasificados.csv`, `resumen_error_analysis.json`.
**Notebook anterior:** `notebooks/pipeline/09b_cierre_modelos_dev.ipynb`.
**Notebook siguiente:** fase clínica externa opcional (`notebooks/analysis/10_validacion_clinica_ips.ipynb`).


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** análisis descriptivo de errores del modelo final congelado.
- **Herramientas/librerías:** `pandas`, `matplotlib`, `CountVectorizer` de `scikit-learn`, métricas de confusión y utilidades de `utils_shared`.
- **Por qué es adecuada aquí:** permite identificar rápidamente errores por clase, casos ambiguos y patrones textuales útiles para discusión clínica sin reabrir el entrenamiento.
- **Limitación:** esto todavía no es xAI formal. Sirve para auditoría clínica inicial, no para atribución rigurosa de importancia por feature o explicación local completa.
- **Alternativa para la fase posterior:** `SHAP`, `LIME` o análisis por familias de features, pero eso pertenece a la etapa de xAI todavía pendiente.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
from IPython.display import display

from utils_shared import setup_paths, ensure_dir, load_splits

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
OUTPUTS_PATH = paths['OUTPUTS_PATH']
SPLITS_PATH = paths['SPLITS_PATH']

def _resolver_contexto_desde_cierre() -> dict | None:
    cierres = sorted(
        [p for p in OUTPUTS_PATH.glob('cierre_modelos_dev_*') if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
    )
    if not cierres:
        return None
    cierre = cierres[-1]
    decision_path = cierre / 'decision_modelo_final.json'
    ranking_path = cierre / 'ranking_modelos_dev.csv'
    if not decision_path.exists() or not ranking_path.exists():
        return None
    try:
        decision = json.loads(decision_path.read_text(encoding='utf-8'))
        variante = ((decision.get('modelo_hibrido_final') or {}).get('modelo_variante') or '').strip()
        if not variante:
            return None
        ranking = pd.read_csv(ranking_path)
        hit = ranking[ranking['modelo_variante'].astype(str) == variante]
        if hit.empty:
            return None
        row = hit.iloc[0]
        return {
            'cierre_dir': cierre,
            'decision': decision,
            'modelo_variante': variante,
            'train_run_id': str(row.get('run_id_train_referencia', '')).strip() or None,
            'perfil': str(row.get('perfil', '')).strip().lower() or None,
            'modelo': str(row.get('modelo', '')).strip().lower() or None,
            'split': str(decision.get('split_decision', 'dev')).strip().lower() or 'dev',
        }
    except Exception:
        return None

CIERRE_INFO = _resolver_contexto_desde_cierre() or {}
TRAIN_RUN_ID = os.getenv('ERROR_TRAIN_RUN_ID') or CIERRE_INFO.get('train_run_id')
ERROR_PROFILE = os.getenv('ERROR_MODEL_PROFILE') or CIERRE_INFO.get('perfil')
ERROR_MODEL = os.getenv('ERROR_MODEL_NAME') or CIERRE_INFO.get('modelo')
ERROR_SPLIT = (os.getenv('ERROR_EVAL_SPLIT') or CIERRE_INFO.get('split') or 'dev').strip().lower()
MODELO_VARIANTE_FINAL = CIERRE_INFO.get('modelo_variante')
if TRAIN_RUN_ID is None:
    cand = sorted([p for p in OUTPUTS_PATH.glob('train_*') if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not cand:
        raise FileNotFoundError('No se encontraron corridas train_* en data/outputs')
    TRAIN_RUN_ID = cand[-1].name

TRAIN_DIR = OUTPUTS_PATH / TRAIN_RUN_ID
ERR_RUN_ID = os.getenv('ERROR_RUN_ID') or pd.Timestamp.now().strftime('error_analysis_%Y%m%d_%H%M%S')
ERR_DIR = ensure_dir(OUTPUTS_PATH / ERR_RUN_ID)
FIG_DIR = ensure_dir(ERR_DIR / 'figures')
print('TRAIN_DIR:', TRAIN_DIR)
print('ERR_DIR:', ERR_DIR)
print('MODELO_VARIANTE_FINAL:', MODELO_VARIANTE_FINAL or 'no_resuelto')
print('ERROR_PROFILE/MODEL/SPLIT:', ERROR_PROFILE, ERROR_MODEL, ERROR_SPLIT)


In [ ]:
# Selección del modelo final congelado en 09b cuando exista cierre formal
pred_path = None
if ERROR_PROFILE and ERROR_MODEL:
    pred_path = TRAIN_DIR / f'predicciones_{ERROR_PROFILE}_{ERROR_MODEL}_{ERROR_SPLIT}.csv'
    if pred_path.exists():
        print('Modelo final congelado:', MODELO_VARIANTE_FINAL or f'{ERROR_PROFILE}|{ERROR_MODEL}')
        print('Archivo de predicciones:', pred_path)
    else:
        print('Aviso: no existe el archivo de predicciones esperado para el modelo final congelado:', pred_path)
        pred_path = None

if pred_path is None:
    cand_comp = sorted(TRAIN_DIR.glob('comparacion_modelos_*.csv'), key=lambda p: p.stat().st_mtime)
    if not cand_comp:
        raise FileNotFoundError(f'No existe comparacion_modelos_*.csv en {TRAIN_DIR}')

    comp_df = pd.read_csv(cand_comp[-1])
    if comp_df.empty:
        raise ValueError('Archivo de comparación vacío.')

    best = comp_df.sort_values(['macro_f1', 'balanced_acc'], ascending=False).iloc[0]
    best_profile = str(best['profile']).lower()
    best_model = str(best['model']).lower()
    best_split = str(best.get('eval_split', 'dev')).lower()
    pred_path = TRAIN_DIR / f'predicciones_{best_profile}_{best_model}_{best_split}.csv'
    if not pred_path.exists():
        pred_files = sorted(TRAIN_DIR.glob('predicciones_*_*.csv'))
        if not pred_files:
            raise FileNotFoundError('No se encontraron predicciones exportadas por 07.')
        pred_path = pred_files[0]
    print('Modelo seleccionado por fallback cuantitativo:', best_profile, best_model, '| split:', best_split)
    print('Archivo de predicciones:', pred_path)


In [ ]:
# Carga de predicciones y unión con texto original
# Se reutilizan las salidas de 07/08 para mantener trazabilidad entre entrenamiento y análisis.

pred_df = pd.read_csv(pred_path)
if not {'row_id', 'y_true', 'y_pred'}.issubset(pred_df.columns):
    raise ValueError('El archivo de predicciones no contiene row_id, y_true y y_pred.')

base_df, _, _, _ = load_splits(SPLITS_PATH)
use_cols = [c for c in ['row_id', 'texto', 'etiqueta', 'patient_id'] if c in base_df.columns]
base_sub = base_df[use_cols].copy()

df = pred_df.merge(base_sub, on='row_id', how='left')
df['error'] = (df['y_true'].astype(str) != df['y_pred'].astype(str)).astype(int)

print('Registros de predicción:', len(df))
print('Tasa de error global:', round(df['error'].mean(), 4))


In [ ]:
# Métricas de error por clase
resumen_clase = (
    df.groupby('y_true')
      .agg(n=('row_id', 'count'), errores=('error', 'sum'))
      .reset_index()
)
resumen_clase['tasa_error'] = resumen_clase['errores'] / resumen_clase['n']

resumen_path = ERR_DIR / 'resumen_error_por_clase.csv'
resumen_clase.to_csv(resumen_path, index=False)
print('Guardado:', resumen_path)
display(resumen_clase.sort_values('tasa_error', ascending=False))


In [ ]:
# Matriz de confusión del modelo seleccionado
labels = sorted(pd.Series(list(set(df['y_true']) | set(df['y_pred']))).astype(str).unique())
cm = confusion_matrix(df['y_true'].astype(str), df['y_pred'].astype(str), labels=labels)
cmn = confusion_matrix(df['y_true'].astype(str), df['y_pred'].astype(str), labels=labels, normalize='true')

cm_csv = ERR_DIR / 'matriz_confusion.csv'
pd.DataFrame(cm, index=labels, columns=labels).to_csv(cm_csv)

plt.figure(figsize=(7, 6))
plt.imshow(cmn)
plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.yticks(range(len(labels)), labels)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de confusión normalizada (modelo seleccionado)')
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
cm_fig = FIG_DIR / 'matriz_confusion.png'
plt.savefig(cm_fig, dpi=200)
plt.show()
print('Guardado:', cm_csv)
print('Guardado:', cm_fig)


In [ ]:
# Términos distintivos en errores (unigramas)
stop_words = [
    'de','la','que','el','en','y','a','los','se','del','las','un','por','con','no','una','su','para','es','al',
    'lo','como','mas','pero','sus','le','ya','o','fue','este','ha','me','si','sin','sobre','tambien','hasta','son'
]

def top_unigramas(textos, top_k=30):
    textos = [t for t in textos if isinstance(t, str) and t.strip()]
    if not textos:
        return pd.DataFrame(columns=['termino', 'conteo'])
    vec = CountVectorizer(ngram_range=(1, 1), stop_words=stop_words)
    bag = vec.fit_transform(textos)
    freqs = np.asarray(bag.sum(axis=0)).ravel()
    terms = np.array(vec.get_feature_names_out())
    idx = np.argsort(freqs)[::-1][:top_k]
    return pd.DataFrame({'termino': terms[idx], 'conteo': freqs[idx]})

rows = []
for clase in sorted(df['y_true'].astype(str).unique()):
    fn = df[(df['y_true'].astype(str) == clase) & (df['y_pred'].astype(str) != clase)]
    fp = df[(df['y_true'].astype(str) != clase) & (df['y_pred'].astype(str) == clase)]

    top_fn = top_unigramas(fn.get('texto', pd.Series(dtype=str)).tolist(), top_k=30)
    top_fn['tipo_error'] = 'FN'
    top_fn['clase_objetivo'] = clase

    top_fp = top_unigramas(fp.get('texto', pd.Series(dtype=str)).tolist(), top_k=30)
    top_fp['tipo_error'] = 'FP'
    top_fp['clase_objetivo'] = clase

    rows.append(top_fn)
    rows.append(top_fp)

terms_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['termino','conteo','tipo_error','clase_objetivo'])
terms_path = ERR_DIR / 'terminos_distintivos_errores.csv'
terms_df.to_csv(terms_path, index=False)
print('Guardado:', terms_path)


In [ ]:
# Casos mal clasificados para revisión clínica
errores_df = df[df['error'] == 1].copy()
cols_show = [c for c in ['row_id', 'y_true', 'y_pred', 'etiqueta', 'texto', 'patient_id'] if c in errores_df.columns]
casos_path = ERR_DIR / 'casos_mal_clasificados.csv'
errores_df[cols_show].to_csv(casos_path, index=False)
print('Guardado:', casos_path)
print('Total casos mal clasificados:', len(errores_df))


In [ ]:
# Resumen de ejecución en JSON
summary = {
    'train_run_id_origen': TRAIN_RUN_ID,
    'predicciones_analizadas': pred_path.name,
    'n_total': int(len(df)),
    'tasa_error_global': float(df['error'].mean()),
}

summary_path = ERR_DIR / 'resumen_error_analysis.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Resumen guardado:', summary_path)
print('Análisis de errores finalizado.')


## Capa clínica adicional para revisión externa

Este notebook exporta el material crudo de error analysis del modelo final en `dev`.
La curación clínica posterior puede organizarse en `10_validacion_clinica_ips.ipynb`, donde los errores se agrupan explícitamente en:

- `ansiedad→depresion`;
- `depresion→ansiedad`;
- casos de frontera ambigua;
- notas de seguimiento o baja especificidad diagnóstica;
- posibles casos que conviene discutir por ruido de etiquetado o por solapamiento clínico.

La idea es separar dos niveles:

1. `09`: trazabilidad y export del error analysis alineado al modelo congelado;
2. `10`: lectura clínica externa y preparación de material reusable para revisión clínica y futura xAI.
